In [ ]:
# Group 15

# Group 15: SMS Spam ClassificationThis project classifies messages as **ham** or **spam**. The messages are cleaned and converted to TF-IDF features. A **linear SVM** (`LinearSVC`) is trained and evaluated using accuracy, error rate, MAE, precision, recall, F1-score, and a confusion matrix.

In [ ]:
import pandas as pd

In [1]:
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive to access the file
drive.mount('/content/drive')

# Path to the file provided (assuming it's in a accessible location or manually downloaded/placed)
# Note: Google Drive file IDs usually require downloading or direct path reference if in 'My Drive'
file_path = '/content/drive/MyDrive/spam.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path, encoding='latin-1')
    # The new file might have different column names, standardizing them to label/message
    df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'message'})
    display(df.head())
else:
    print(f'File not found at {file_path}. Please check the path.')

Mounted at /content/drive


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [3]:
df.describe()

,label,message
count,5572,5572
unique,2,5169
top,ham,"Sorry, I'll call later"
freq,4825,30


In [4]:
import nltk
from nltk.corpus import stopwords
import string

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.lower()
    words = [word for word in text.split() if word not in stop_words]
    return " ".join(words)

df['clean_message'] = df['message'].apply(preprocess_text)
display(df[['message', 'clean_message']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['clean_message'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
from sklearn.svm import LinearSVC
model = LinearSVC(random_state=42)
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       965
        spam       0.98      0.77      0.87       150

    accuracy                           0.97      1115
   macro avg       0.97      0.89      0.92      1115
weighted avg       0.97      0.97      0.97      1115



In [ ]:
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report, confusion_matrix

y_pred = model.predict(X_test)
error_rate = 1 - accuracy_score(y_test, y_pred)
y_test_num = (y_test == 'spam').astype(int)
y_pred_num = (y_pred == 'spam').astype(int)
mae = mean_absolute_error(y_test_num, y_pred_num)

print(classification_report(y_test, y_pred))
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Error rate:', round(error_rate, 4))
print('MAE:', round(mae, 4))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

In [16]:
def predict_message(message):
    cleaned = preprocess_text(message)
    vectorized = tfidf.transform([cleaned])
    return model.predict(vectorized)[0]

print(f"Prediction: {predict_message('sssf ggghni hone. ze now.').upper()}")

Prediction: HAM


In [17]:
import joblib
joblib.dump(model, 'spam_detector_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("Saved!")

Saved!
